# 05B · Evaluación del escenario de siniestros con víctimas
### Fase 5 de CRISP-DM

Este notebook evalúa los modelos entrenados con siniestros `Con Heridos` o `Con Muertos`. Random Forest fue seleccionado mediante validación temporal expansiva dentro de 2018–2022 y utiliza el umbral candidato 0,55. No se reentrenan ni sobrescriben modelos.

## Alcance

La evaluación 2023–2024 es retrospectiva: esos años ya fueron inspeccionados durante el desarrollo. La etiqueta es híbrida: en 49 de 80 combinaciones localidad–franja, `Alto_Riesgo=1` significa que ocurrió al menos un siniestro con víctimas; en 28 exige dos o más y en tres exige tres o más. Las métricas evalúan predicción respecto a esta etiqueta, no peligrosidad causal ni riesgo individual.

## 1. Importación y configuración

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
DATA_FILE = Path('../data/processed/dataset_victimas_localidad_franja_fecha.parquet')
MODELS_PATH = Path('../models/victimas')
REPORTS_PATH = Path('../reports/evaluation_victimas')
REPORTS_PATH.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## 2. Datos y artefactos

In [ ]:
dataset = pd.read_parquet(DATA_FILE)
dataset['Fecha_Acc'] = pd.to_datetime(dataset['Fecha_Acc'])
metadata = json.loads((MODELS_PATH / 'metadata_modelo.json').read_text(encoding='utf-8'))
feature_cols = metadata['variables']
selected_model = metadata['modelo_seleccionado']
selected_threshold = float(metadata['umbral_candidato'])
pipelines = {
    'Regresión Logística': joblib.load(MODELS_PATH / 'pipeline_logistica_victimas.pkl'),
    'Random Forest': joblib.load(MODELS_PATH / 'pipeline_random_forest_victimas.pkl'),
    'XGBoost': joblib.load(MODELS_PATH / 'pipeline_xgboost_victimas.pkl'),
}
required = feature_cols + ['Fecha_Acc', 'Localidad', 'Franja_Horaria', 'Periodo', 'Alto_Riesgo']
assert dataset[required].isna().sum().sum() == 0
assert selected_model == 'Random Forest'
print('Modelo seleccionado:', selected_model, '| Umbral:', selected_threshold)
print('Dataset:', dataset.shape)

In [ ]:
test = dataset[dataset.Periodo.eq('test')].copy()
test['Anio'] = test.Fecha_Acc.dt.year
X_test, y_test = test[feature_cols], test.Alto_Riesgo.astype(int)
scores = {name: pipeline.predict_proba(X_test)[:, 1] for name, pipeline in pipelines.items()}
thresholds = {name: selected_threshold if name == selected_model else 0.5 for name in pipelines}
predictions = {name: (scores[name] >= thresholds[name]).astype(int) for name in pipelines}
print(f'Test: {len(test):,} observaciones; prevalencia={y_test.mean():.2%}')

## 3. Métricas globales

In [ ]:
def classification_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'AUC_ROC': roc_auc_score(y_true, y_score),
        'Average_Precision': average_precision_score(y_true, y_score),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Especificidad': tn / (tn + fp),
        'Balanced_Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Brier': brier_score_loss(y_true, y_score),
        'Tasa_Predicha_Positiva': np.mean(y_pred),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }

global_metrics = pd.DataFrame([
    {'Modelo': name, 'Umbral': thresholds[name],
     **classification_metrics(y_test, predictions[name], scores[name])}
    for name in pipelines
]).sort_values('Average_Precision', ascending=False)
global_metrics.to_csv(REPORTS_PATH / 'metricas_globales.csv', index=False, encoding='utf-8-sig')
global_metrics.round(4)

## 4. Matrices de confusión

In [ ]:
model_names = list(pipelines)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, predictions[name], labels=[0, 1])
    ax.imshow(cm, cmap='Blues')
    for row in range(2):
        for col in range(2):
            ax.text(col, row, f'{cm[row, col]:,}', ha='center', va='center',
                    color='white' if cm[row, col] > cm.max()/2 else 'black')
    ax.set(title=f'{name} (u={thresholds[name]:.2f})', xlabel='Predicción', ylabel='Valor real')
    ax.set_xticks([0, 1], ['No alto', 'Alto']); ax.set_yticks([0, 1], ['No alto', 'Alto'])
plt.suptitle('Matrices de confusión — escenario con víctimas', y=1.03)
plt.tight_layout(); plt.savefig(REPORTS_PATH / '01_matrices_confusion.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Curvas ROC y Precision-Recall

In [ ]:
colors = {'Regresión Logística': '#4472C4', 'Random Forest': '#70AD47', 'XGBoost': '#ED7D31'}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in model_names:
    fpr, tpr, _ = roc_curve(y_test, scores[name])
    precision, recall, _ = precision_recall_curve(y_test, scores[name])
    axes[0].plot(fpr, tpr, color=colors[name], label=f'{name} (AUC={roc_auc_score(y_test, scores[name]):.3f})')
    axes[1].plot(recall, precision, color=colors[name], label=f'{name} (AP={average_precision_score(y_test, scores[name]):.3f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Azar')
axes[1].axhline(y_test.mean(), linestyle='--', color='gray', label=f'Prevalencia ({y_test.mean():.3f})')
axes[0].set(title='Curva ROC', xlabel='Falsos positivos', ylabel='Verdaderos positivos')
axes[1].set(title='Curva Precision-Recall', xlabel='Recall', ylabel='Precisión')
for ax in axes: ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(REPORTS_PATH / '02_curvas_roc_pr.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. Calibración

In [ ]:
calibration_rows = []
plt.figure(figsize=(7, 6))
for name in model_names:
    observed, predicted = calibration_curve(y_test, scores[name], n_bins=10, strategy='quantile')
    plt.plot(predicted, observed, marker='o', color=colors[name],
             label=f'{name} (Brier={brier_score_loss(y_test, scores[name]):.3f})')
    ranks = pd.qcut(pd.Series(scores[name]), q=10, labels=False, duplicates='drop')
    temp = pd.DataFrame({'Grupo': ranks, 'Score': scores[name], 'Real': y_test.to_numpy()})
    grouped = temp.groupby('Grupo').agg(Score_Medio=('Score','mean'),
                                         Frecuencia_Observada=('Real','mean'), Observaciones=('Real','size')).reset_index()
    grouped.insert(0, 'Modelo', name); calibration_rows.append(grouped)
plt.plot([0,1], [0,1], '--', color='gray', label='Calibración perfecta')
plt.xlabel('Score medio'); plt.ylabel('Frecuencia observada'); plt.title('Calibración 2023–2024')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(REPORTS_PATH / '03_calibracion.png', dpi=160, bbox_inches='tight'); plt.show()
calibration_table = pd.concat(calibration_rows, ignore_index=True)
calibration_table.to_csv(REPORTS_PATH / 'calibracion_por_deciles.csv', index=False, encoding='utf-8-sig')
calibration_table.round(4)

Los modelos utilizan balanceo de clases; por ello `predict_proba` no debe asumirse automáticamente como una probabilidad calibrada. Si la curva se aleja de la diagonal, el dashboard debe presentar puntuaciones relativas o niveles de priorización.

## 7. Estabilidad entre 2023 y 2024

In [ ]:
year_rows = []
for year in [2023, 2024]:
    mask = test.Anio.eq(year).to_numpy()
    for name in model_names:
        year_rows.append({'Anio': year, 'Modelo': name, 'Umbral': thresholds[name],
                          **classification_metrics(y_test.to_numpy()[mask], predictions[name][mask], scores[name][mask])})
year_metrics = pd.DataFrame(year_rows)
year_metrics.to_csv(REPORTS_PATH / 'metricas_por_anio.csv', index=False, encoding='utf-8-sig')
year_metrics[['Anio','Modelo','F1','AUC_ROC','Average_Precision','Precision','Recall','Brier']].round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['F1','Average_Precision','Recall']):
    pivot = year_metrics.pivot(index='Anio', columns='Modelo', values=metric)
    pivot[model_names].plot(kind='bar', ax=ax, color=[colors[n] for n in model_names])
    ax.set_title(metric.replace('_',' ')); ax.tick_params(axis='x', rotation=0)
    if ax is not axes[0]: ax.get_legend().remove()
axes[0].legend(fontsize=8); plt.tight_layout()
plt.savefig(REPORTS_PATH / '04_metricas_por_anio.png', dpi=160, bbox_inches='tight'); plt.show()

## 8. Errores de Random Forest por localidad y franja

In [ ]:
rf_eval = test[['Fecha_Acc','Anio','Localidad','Franja_Horaria','Alto_Riesgo']].copy()
rf_eval['Score'] = scores[selected_model]
rf_eval['Prediccion'] = predictions[selected_model]

def subgroup_metrics(frame, column):
    rows = []
    for value, part in frame.groupby(column):
        result = classification_metrics(part.Alto_Riesgo, part.Prediccion, part.Score)
        rows.append({column: value, 'Observaciones': len(part), 'Positivos': int(part.Alto_Riesgo.sum()),
                     'Prevalencia': part.Alto_Riesgo.mean(), **result})
    return pd.DataFrame(rows)

by_locality = subgroup_metrics(rf_eval, 'Localidad').sort_values('F1', ascending=False)
by_slot = subgroup_metrics(rf_eval, 'Franja_Horaria').sort_values('F1', ascending=False)
by_locality.to_csv(REPORTS_PATH / 'metricas_random_forest_por_localidad.csv', index=False, encoding='utf-8-sig')
by_slot.to_csv(REPORTS_PATH / 'metricas_random_forest_por_franja.csv', index=False, encoding='utf-8-sig')
display(by_locality.round(4)); display(by_slot.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
loc_plot = by_locality.sort_values('F1')
axes[0].barh(loc_plot.Localidad, loc_plot.F1, color='#70AD47'); axes[0].set_title('F1 por localidad')
slot_plot = by_slot.set_index('Franja_Horaria')
slot_plot[['FP','FN']].plot(kind='bar', ax=axes[1], color=['#ED7D31','#C00000'])
axes[1].set_title('Errores por franja'); axes[1].tick_params(axis='x', rotation=0)
slot_plot['Recall'].plot(kind='bar', ax=axes[2], color='#4472C4')
axes[2].set_title('Recall por franja'); axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig(REPORTS_PATH / '05_errores_territoriales_temporales.png', dpi=160, bbox_inches='tight')
plt.show()

Las diferencias entre grupos deben interpretarse junto con la prevalencia y el número de positivos. No constituyen un ranking de peligrosidad y pueden reflejar umbrales de etiqueta diferentes.

## 9. Promedios macro por localidad

In [ ]:
rf_global = global_metrics.set_index('Modelo').loc[selected_model]
macro_rows = []
for metric in ['F1','Precision','Recall','AUC_ROC','Average_Precision']:
    macro_rows.append({'Metrica': metric, 'Global': rf_global[metric],
                       'Macro_Localidades': by_locality[metric].mean()})
macro_metrics = pd.DataFrame(macro_rows)
macro_metrics.to_csv(REPORTS_PATH / 'metricas_macro_localidad.csv', index=False, encoding='utf-8-sig')
macro_metrics.round(4)

## 10. Importancia interna de Random Forest

In [ ]:
rf_pipeline = pipelines[selected_model]
transformed_names = rf_pipeline.named_steps['preprocesador'].get_feature_names_out()
internal_importance = pd.DataFrame({
    'Variable': transformed_names,
    'Importancia': rf_pipeline.named_steps['modelo'].feature_importances_
}).sort_values('Importancia', ascending=False)
internal_importance.to_csv(REPORTS_PATH / 'importancia_interna_random_forest.csv', index=False, encoding='utf-8-sig')
display(internal_importance.head(20).round(4))
top = internal_importance.head(20).sort_values('Importancia')
plt.figure(figsize=(9,7)); plt.barh(top.Variable, top.Importancia, color='#70AD47')
plt.title('Importancia interna de Random Forest'); plt.xlabel('Importancia'); plt.tight_layout()
plt.savefig(REPORTS_PATH / '06_importancia_interna_random_forest.png', dpi=160, bbox_inches='tight'); plt.show()

## 11. Importancia por permutación

In [ ]:
sample = test.sample(n=min(5000, len(test)), random_state=RANDOM_STATE)
permutation = permutation_importance(
    rf_pipeline, sample[feature_cols], sample.Alto_Riesgo,
    scoring='average_precision', n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1
)
permutation_table = pd.DataFrame({
    'Variable': feature_cols, 'Caida_AP_Media': permutation.importances_mean,
    'Desviacion': permutation.importances_std
}).sort_values('Caida_AP_Media', ascending=False)
permutation_table.to_csv(REPORTS_PATH / 'importancia_permutacion_random_forest.csv', index=False, encoding='utf-8-sig')
display(permutation_table.round(5))
plot_perm = permutation_table.sort_values('Caida_AP_Media')
plt.figure(figsize=(9,6)); plt.barh(plot_perm.Variable, plot_perm.Caida_AP_Media, xerr=plot_perm.Desviacion, color='#4472C4')
plt.title('Importancia por permutación sobre Average Precision'); plt.xlabel('Caída media de AP')
plt.tight_layout(); plt.savefig(REPORTS_PATH / '07_importancia_permutacion.png', dpi=160, bbox_inches='tight'); plt.show()

La importancia interna y la permutación no representan causalidad ni porcentaje de accidentes explicado. Variables correlacionadas pueden compartir importancia; la permutación puede crear combinaciones poco realistas.

## 12. Brier frente a líneas base constantes

In [ ]:
train_prevalence = dataset.loc[dataset.Periodo.eq('train'), 'Alto_Riesgo'].mean()
test_prevalence = y_test.mean()
constant_train = np.full(len(y_test), train_prevalence)
constant_oracle = np.full(len(y_test), test_prevalence)
brier_comparison = pd.DataFrame([
    {'Escenario': 'Random Forest', 'Score_Medio': scores[selected_model].mean(),
     'Brier': brier_score_loss(y_test, scores[selected_model]),
     'AUC_ROC': roc_auc_score(y_test, scores[selected_model]),
     'Average_Precision': average_precision_score(y_test, scores[selected_model])},
    {'Escenario': 'Constante prevalencia train', 'Score_Medio': train_prevalence,
     'Brier': brier_score_loss(y_test, constant_train), 'AUC_ROC': 0.5,
     'Average_Precision': test_prevalence},
    {'Escenario': 'Constante prevalencia test (referencia retrospectiva)', 'Score_Medio': test_prevalence,
     'Brier': brier_score_loss(y_test, constant_oracle), 'AUC_ROC': 0.5,
     'Average_Precision': test_prevalence}
])
brier_comparison.to_csv(REPORTS_PATH / 'comparacion_brier_lineas_base.csv', index=False, encoding='utf-8-sig')
brier_comparison.round(4)

Random Forest puede superar la referencia en AUC-ROC y Average Precision y, simultáneamente, obtener peor Brier. Esto significa que ordena los casos, pero sus valores crudos no representan bien las frecuencias observadas. La constante basada en test se incluye solo como referencia retrospectiva ideal y no estaría disponible al predecir el futuro.

## 13. Distribución de scores y volumen histórico por localidad

In [ ]:
train = dataset[dataset.Periodo.eq('train')].copy()
historical_volume = train.groupby('Localidad').Num_Accidentes.sum().rename('Volumen_Historico_2018_2022')
score_summary = (rf_eval.groupby('Localidad').Score
                 .agg(Score_Min='min', Score_Medio='mean', Score_Mediana='median',
                      Score_P95=lambda x: x.quantile(.95), Score_Max='max').reset_index())
score_summary['Alertas_Umbral_055'] = rf_eval.groupby('Localidad').Prediccion.sum().values
locality_diagnostic = (by_locality.merge(score_summary, on='Localidad')
                       .merge(historical_volume.reset_index(), on='Localidad'))
locality_diagnostic.to_csv(REPORTS_PATH / 'diagnostico_scores_localidad.csv', index=False, encoding='utf-8-sig')
locality_diagnostic.sort_values('Score_Max').round(4)

In [ ]:
correlation_columns = ['Volumen_Historico_2018_2022','Prevalencia','Score_Medio','Recall','F1']
locality_correlations = locality_diagnostic[correlation_columns].corr(method='spearman')
locality_correlations.to_csv(REPORTS_PATH / 'correlaciones_spearman_localidad.csv', encoding='utf-8-sig')
display(locality_correlations.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(locality_diagnostic.Volumen_Historico_2018_2022, locality_diagnostic.Recall, color='#4472C4')
axes[0].set(xlabel='Siniestros con víctimas 2018–2022', ylabel='Recall', title='Volumen histórico frente a recall')
axes[1].scatter(locality_diagnostic.Prevalencia, locality_diagnostic.Recall, color='#ED7D31')
axes[1].set(xlabel='Prevalencia 2023–2024', ylabel='Recall', title='Prevalencia frente a recall')
for _, row in locality_diagnostic.iterrows():
    if row.Localidad in ['CANDELARIA','SUMAPAZ','SANTA FE']:
        axes[0].annotate(row.Localidad.title(), (row.Volumen_Historico_2018_2022, row.Recall), fontsize=8)
        axes[1].annotate(row.Localidad.title(), (row.Prevalencia, row.Recall), fontsize=8)
plt.tight_layout(); plt.savefig(REPORTS_PATH / '08_volumen_prevalencia_recall.png', dpi=160, bbox_inches='tight')
plt.show()

La correlación entre volumen histórico y recall determina si el problema de Candelaria puede generalizarse. Una correlación cercana a cero impide afirmar que el modelo falle sistemáticamente en todas las localidades de bajo volumen; Candelaria y Sumapaz deben tratarse como casos críticos particulares.

## 14. Diagnóstico específico de Candelaria

In [ ]:
candelaria = rf_eval[rf_eval.Localidad.eq('CANDELARIA')].copy()
candelaria_summary = pd.DataFrame([{
    'Observaciones': len(candelaria), 'Positivos': int(candelaria.Alto_Riesgo.sum()),
    'Prevalencia': candelaria.Alto_Riesgo.mean(), 'Score_Medio': candelaria.Score.mean(),
    'Score_Mediana': candelaria.Score.median(), 'Score_P95': candelaria.Score.quantile(.95),
    'Score_Max': candelaria.Score.max(), 'Alertas_055': int(candelaria.Prediccion.sum())
}])
candelaria_summary.to_csv(REPORTS_PATH / 'resumen_candelaria.csv', index=False, encoding='utf-8-sig')
candelaria_summary.round(4)

In [ ]:
candelaria_threshold_rows = []
for threshold in np.linspace(0.05, 0.55, 101):
    pred = (candelaria.Score.to_numpy() >= threshold).astype(int)
    candelaria_threshold_rows.append({'Umbral': threshold,
        **classification_metrics(candelaria.Alto_Riesgo, pred, candelaria.Score)})
candelaria_thresholds = pd.DataFrame(candelaria_threshold_rows)
candelaria_best_threshold = float(candelaria_thresholds.loc[candelaria_thresholds.F1.idxmax(), 'Umbral'])
candelaria_thresholds.to_csv(REPORTS_PATH / 'diagnostico_umbral_candelaria.csv', index=False, encoding='utf-8-sig')

global_default = classification_metrics(y_test, predictions[selected_model], scores[selected_model])
global_low_pred = (scores[selected_model] >= candelaria_best_threshold).astype(int)
global_low = classification_metrics(y_test, global_low_pred, scores[selected_model])
targeted_pred = predictions[selected_model].copy()
cand_mask = test.Localidad.eq('CANDELARIA').to_numpy()
targeted_pred[cand_mask] = (scores[selected_model][cand_mask] >= candelaria_best_threshold).astype(int)
targeted = classification_metrics(y_test, targeted_pred, scores[selected_model])
threshold_impact = pd.DataFrame({
    'Umbral global 0.55': global_default,
    f'Umbral global {candelaria_best_threshold:.3f}': global_low,
    f'Umbral {candelaria_best_threshold:.3f} solo Candelaria': targeted
}).T
threshold_impact.to_csv(REPORTS_PATH / 'impacto_umbral_candelaria.csv', encoding='utf-8-sig')
print(f'Umbral que maximiza F1 en Candelaria, calculado retrospectivamente: {candelaria_best_threshold:.3f}')
display(threshold_impact[['F1','Precision','Recall','FP','FN','TP','TN']].round(4))

El umbral específico se calcula sobre test y es solamente diagnóstico. No debe utilizarse en producción. Sirve para cuantificar cuántas falsas alertas aparecerían al intentar recuperar Candelaria mediante un umbral global y para motivar una futura selección de umbrales territoriales exclusivamente con validación temporal.

## 15. Calibración y Brier por localidad

In [ ]:
train_prevalence_locality = train.groupby('Localidad').Alto_Riesgo.mean()
local_calibration_rows = []
for locality, part in rf_eval.groupby('Localidad'):
    local_base = float(train_prevalence_locality.loc[locality])
    local_calibration_rows.append({
        'Localidad': locality, 'Observaciones': len(part), 'Positivos': int(part.Alto_Riesgo.sum()),
        'Prevalencia_Train': local_base, 'Prevalencia_Test': part.Alto_Riesgo.mean(),
        'Score_Medio_RF': part.Score.mean(), 'Brier_RF': brier_score_loss(part.Alto_Riesgo, part.Score),
        'Brier_Constante_Train_Local': brier_score_loss(part.Alto_Riesgo, np.full(len(part), local_base))
    })
local_calibration = pd.DataFrame(local_calibration_rows)
local_calibration['Diferencia_Brier_RF_Menos_Base'] = (
    local_calibration.Brier_RF - local_calibration.Brier_Constante_Train_Local
)
local_calibration.to_csv(REPORTS_PATH / 'calibracion_por_localidad.csv', index=False, encoding='utf-8-sig')
local_calibration.sort_values('Diferencia_Brier_RF_Menos_Base', ascending=False).round(4)

## 16. Volumen histórico y desempeño por franja

In [ ]:
slot_volume = train.groupby('Franja_Horaria').Num_Accidentes.sum().rename('Volumen_Historico_2018_2022')
slot_scores = rf_eval.groupby('Franja_Horaria').Score.mean().rename('Score_Medio')
slot_diagnostic = by_slot.merge(slot_volume.reset_index(), on='Franja_Horaria').merge(slot_scores.reset_index(), on='Franja_Horaria')
slot_diagnostic.to_csv(REPORTS_PATH / 'diagnostico_volumen_franja.csv', index=False, encoding='utf-8-sig')
slot_diagnostic[['Franja_Horaria','Volumen_Historico_2018_2022','Prevalencia','Score_Medio','F1','Precision','Recall']].round(4)

La madrugada combina menor volumen histórico, menor score medio y menor recall. El patrón es compatible con una señal histórica más débil, pero cuatro franjas no permiten demostrar causalidad ni establecer una relación general entre volumen y desempeño.

## 17. Exportación de predicciones y conclusión ejecutiva

In [ ]:
rf_eval.to_parquet(REPORTS_PATH / 'predicciones_random_forest_2023_2024.parquet', index=False)
rf = global_metrics.set_index('Modelo').loc[selected_model]
best_loc, worst_loc = by_locality.iloc[0], by_locality.iloc[-1]
best_slot, worst_slot = by_slot.iloc[0], by_slot.iloc[-1]
executive = f"""# Conclusión ejecutiva del escenario con víctimas

Random Forest, seleccionado con validación temporal 2018–2022 y umbral {selected_threshold:.2f}, obtiene en 2023–2024 F1={rf.F1:.4f}, AUC-ROC={rf.AUC_ROC:.4f}, Average Precision={rf.Average_Precision:.4f}, precisión={rf.Precision:.2%} y recall={rf.Recall:.2%}. Detecta {int(rf.TP):,} positivos, genera {int(rf.FP):,} falsas alertas y omite {int(rf.FN):,} positivos.

El Brier de Random Forest es {rf.Brier:.4f}, peor que {brier_comparison.iloc[1].Brier:.4f} de la constante basada en la prevalencia de entrenamiento. El modelo supera la referencia en discriminación y ranking, pero no en exactitud probabilística; sus scores no deben mostrarse como probabilidades literales.

El desempeño territorial no es uniforme: {best_loc.Localidad} presenta el mayor F1 ({best_loc.F1:.4f}) y {worst_loc.Localidad} el menor ({worst_loc.F1:.4f}); estos valores deben leerse junto con prevalencia y cantidad de positivos. Por franja, {best_slot.Franja_Horaria} obtiene el mayor F1 ({best_slot.F1:.4f}) y {worst_slot.Franja_Horaria} el menor ({worst_slot.F1:.4f}).

Candelaria contiene {int(candelaria_summary.iloc[0].Positivos)} positivos, pero su score máximo es {candelaria_summary.iloc[0].Score_Max:.4f}, por debajo de 0,55 y también de 0,50. Este es un fallo territorial específico. La correlación de Spearman entre volumen histórico y recall es {locality_correlations.loc['Volumen_Historico_2018_2022','Recall']:.3f}, por lo que no se puede generalizar que todas las localidades con bajo volumen tengan peor recall.

La madrugada presenta el menor volumen histórico, score medio y recall ({by_slot.set_index('Franja_Horaria').loc['Madrugada','Recall']:.2%}). Esta coincidencia es descriptiva y compatible con menor señal histórica, pero no demuestra causalidad. La etiqueta combina ocurrencia y superación de umbral según localidad y franja. Los resultados son retrospectivos, no causales ni confirmatorios, y el sistema no se considera listo para producción sin una evaluación futura independiente y un criterio operativo para falsas alertas y omisiones.
"""
(REPORTS_PATH / 'conclusion_ejecutiva.md').write_text(executive, encoding='utf-8')
print(executive)
print('Reportes guardados en:', REPORTS_PATH.resolve())